# Computer Vision Lab 08 — Image Classification, Camera Calibration & Video Analytics
**BS(AI) 7th Semester**

**Name:** Qasim Mustafa  **Enrollment:** 01-136232-079  **Class:** BS(AI)-7A

| Task | Topic |
|---|---|
| 1 | Image dataset pipeline & classical feature classification (HSV histogram + HOG → SVM / Random Forest) |
| 2 | Camera calibration, distortion correction & perspective transformation (homography) |
| 3 | Video stream analysis — MOG2 motion detection, morphology, contour detection & centroid tracking |

> Run with **Runtime → Run all**. Every dataset is downloaded automatically (TensorFlow *flower_photos*, OpenCV sample checkerboard images, and OpenCV `vtest.avi` surveillance video), so nothing needs to be uploaded.

In [ ]:
# ===================== Setup =====================
!pip -q install scikit-image

import os, glob, time, random, base64, urllib.request, tarfile
from collections import deque, OrderedDict

import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)
from IPython.display import HTML, display

SEED = 42
random.seed(SEED); np.random.seed(SEED)
plt.rcParams['figure.dpi'] = 100
os.makedirs('data', exist_ok=True); os.makedirs('outputs', exist_ok=True)

def download(url, path):
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    return path

def show(img_bgr, title='', ax=None):
    ax = ax or plt.gca()
    if img_bgr.ndim == 2: ax.imshow(img_bgr, cmap='gray')
    else: ax.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    ax.set_title(title); ax.axis('off')

print('OpenCV', cv2.__version__, '| NumPy', np.__version__)

---
# Task 1 — Image Dataset Pipeline & Classical Feature Classification

**Dataset:** TensorFlow *flower_photos* (real RGB photographs). We use **4 classes** (daisy, dandelion, roses, sunflowers) and randomly sample **100 images per class** (400 total).

**Pipeline:** load → resize to **128×128** → **80/20 stratified split** → feature extraction (HSV colour histogram & HOG) → SVM / Random Forest → accuracy + confusion matrix.

*To use your own dataset instead, set `CUSTOM_DATASET_DIR` to a folder containing one sub-folder per class.*

In [ ]:
# ---------- 1.1 Prepare dataset ----------
CUSTOM_DATASET_DIR = None          # e.g. '/content/my_dataset' (one sub-folder per class)
CLASSES = ['daisy', 'dandelion', 'roses', 'sunflowers']
IMAGES_PER_CLASS = 100
IMG_SIZE = (128, 128)

if CUSTOM_DATASET_DIR:
    root = CUSTOM_DATASET_DIR
    CLASSES = sorted(d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d)))
else:
    tgz = download('https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz',
                   'data/flower_photos.tgz')
    if not os.path.isdir('data/flower_photos'):
        with tarfile.open(tgz) as t: t.extractall('data')
    root = 'data/flower_photos'

X_img, y = [], []
for label, cls in enumerate(CLASSES):
    files = sorted(glob.glob(os.path.join(root, cls, '*')))
    files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    random.Random(SEED).shuffle(files)
    loaded = 0
    for f in files:
        img = cv2.imread(f)
        if img is None: continue
        X_img.append(cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_AREA))
        y.append(label); loaded += 1
        if loaded == IMAGES_PER_CLASS: break
    print(f'{cls:12s}: {loaded} images')

X_img, y = np.array(X_img), np.array(y)
print('Dataset tensor:', X_img.shape, '| labels:', y.shape)

In [ ]:
# ---------- 1.2 Visualise samples ----------
fig, axes = plt.subplots(len(CLASSES), 6, figsize=(12, 2.2 * len(CLASSES)))
for r, cls in enumerate(CLASSES):
    idx = np.where(y == r)[0][:6]
    for c, i in enumerate(idx):
        show(X_img[i], cls if c == 0 else '', axes[r, c])
plt.suptitle(f'Sample images (resized to {IMG_SIZE[0]}x{IMG_SIZE[1]})', fontsize=14)
plt.tight_layout(); plt.savefig('outputs/task1_samples.png'); plt.show()

In [ ]:
# ---------- 1.3 80/20 stratified train/test split ----------
X_train_img, X_test_img, y_train, y_test = train_test_split(
    X_img, y, test_size=0.20, stratify=y, random_state=SEED)

print('Train:', X_train_img.shape[0], ' Test:', X_test_img.shape[0])
print('Train per class:', dict(zip(CLASSES, np.bincount(y_train))))
print('Test  per class:', dict(zip(CLASSES, np.bincount(y_test))))

### Feature extraction
* **HSV colour histogram** — the image is converted to HSV and a 3-D histogram with 8×8×8 bins (H, S, V) is computed → **512-D** vector, L1-normalised so it is independent of image size. HSV separates colour (hue) from brightness, which makes it more robust to lighting than RGB.
* **HOG (Histogram of Oriented Gradients)** — grayscale image, 9 orientation bins, 16×16-pixel cells, 2×2-cell blocks with L2-Hys normalisation → **1764-D** vector describing local shape/edge structure.
* **HSV + HOG** — the two vectors concatenated (2276-D) to combine colour and shape cues.

In [ ]:
# ---------- 1.4 Feature extractors ----------
def hsv_histogram(img_bgr, bins=(8, 8, 8)):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, bins, [0, 180, 0, 256, 0, 256])
    hist = hist.flatten()
    return hist / (hist.sum() + 1e-7)

def hog_features(img_bgr, visualize=False):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    return hog(gray, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2),
               block_norm='L2-Hys', feature_vector=True, visualize=visualize)

def extract(imgs, kind):
    feats = []
    for im in imgs:
        if kind == 'HSV':   f = hsv_histogram(im)
        elif kind == 'HOG': f = hog_features(im)
        else:               f = np.hstack([hsv_histogram(im), hog_features(im)])
        feats.append(f)
    return np.array(feats, dtype=np.float32)

features = {}
for kind in ['HSV', 'HOG', 'HSV+HOG']:
    features[kind] = (extract(X_train_img, kind), extract(X_test_img, kind))
    print(f'{kind:8s} feature vector length = {features[kind][0].shape[1]}')

In [ ]:
# ---------- 1.5 Visualise the features for one image ----------
sample = X_train_img[0]
_, hog_img = hog_features(sample, visualize=True)
hsv = cv2.cvtColor(sample, cv2.COLOR_BGR2HSV)

fig, ax = plt.subplots(1, 4, figsize=(16, 3.6))
show(sample, f'Original ({CLASSES[y_train[0]]})', ax[0])
ax[1].imshow(hog_img, cmap='inferno'); ax[1].set_title('HOG visualisation'); ax[1].axis('off')
for ch, (name, rng, col) in enumerate([('Hue', 180, 'tab:orange'), ('Saturation', 256, 'tab:green')]):
    h = cv2.calcHist([hsv], [ch], None, [32], [0, rng]).ravel()
    ax[2 + ch].bar(np.arange(32), h, color=col); ax[2 + ch].set_title(f'{name} histogram (32 bins)')
plt.tight_layout(); plt.savefig('outputs/task1_features.png'); plt.show()

### Classifiers
* **SVM** (RBF kernel, C = 10) inside a pipeline with `StandardScaler`, because SVMs are sensitive to feature scale.
* **Random Forest** (300 trees) — scale-invariant ensemble of decision trees.

Each classifier is trained on each feature set; the best combination is evaluated in detail.

In [ ]:
# ---------- 1.6 Train & evaluate classifiers ----------
models = {
    'SVM (RBF)':     lambda: make_pipeline(StandardScaler(), SVC(kernel='rbf', C=10, gamma='scale', random_state=SEED)),
    'Random Forest': lambda: RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
}

results, trained = [], {}
for feat_name, (Ftr, Fte) in features.items():
    for model_name, make in models.items():
        clf = make().fit(Ftr, y_train)
        pred = clf.predict(Fte)
        acc = accuracy_score(y_test, pred)
        results.append((feat_name, model_name, acc))
        trained[(feat_name, model_name)] = (clf, pred)

print(f"{'Features':10s} | {'Classifier':14s} | Test accuracy")
print('-' * 44)
for f, m, a in results:
    print(f'{f:10s} | {m:14s} | {a*100:6.2f} %')

best_feat, best_model, best_acc = max(results, key=lambda r: r[2])
print(f'\nBest: {best_feat} + {best_model}  ->  accuracy = {best_acc*100:.2f} %')

In [ ]:
# ---------- 1.7 Accuracy comparison chart ----------
feat_names = list(features.keys()); model_names = list(models.keys())
acc_mat = np.array([[next(a for f, m, a in results if f == fn and m == mn) for fn in feat_names] for mn in model_names])
x = np.arange(len(feat_names)); w = 0.38
fig, ax = plt.subplots(figsize=(8, 4))
for i, mn in enumerate(model_names):
    bars = ax.bar(x + (i - 0.5) * w, acc_mat[i] * 100, w, label=mn)
    ax.bar_label(bars, fmt='%.1f')
ax.set_xticks(x, feat_names); ax.set_ylabel('Test accuracy (%)'); ax.set_ylim(0, 100)
ax.set_title('Classifier accuracy by feature type'); ax.legend()
plt.tight_layout(); plt.savefig('outputs/task1_accuracy.png'); plt.show()

In [ ]:
# ---------- 1.8 Confusion matrix & report for the best model ----------
clf, pred = trained[(best_feat, best_model)]
cm = confusion_matrix(y_test, pred)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax[0], cmap='Blues', colorbar=False)
ax[0].set_title(f'Confusion matrix (counts)\n{best_feat} + {best_model} — acc {best_acc*100:.1f}%')
ConfusionMatrixDisplay(cm / cm.sum(1, keepdims=True), display_labels=CLASSES).plot(
    ax=ax[1], cmap='Greens', colorbar=False, values_format='.2f')
ax[1].set_title('Confusion matrix (row-normalised = per-class recall)')
plt.tight_layout(); plt.savefig('outputs/task1_confusion_matrix.png'); plt.show()

print(f'Overall accuracy: {accuracy_score(y_test, pred)*100:.2f} %\n')
print(classification_report(y_test, pred, target_names=CLASSES, digits=3))

In [ ]:
# ---------- 1.9 Example predictions ----------
idx = np.random.RandomState(1).choice(len(X_test_img), 10, replace=False)
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, i in zip(axes.ravel(), idx):
    ok = pred[i] == y_test[i]
    show(X_test_img[i], f'T: {CLASSES[y_test[i]]}\nP: {CLASSES[pred[i]]}', ax)
    ax.title.set_color('green' if ok else 'red')
plt.suptitle('Test-set predictions (green = correct, red = wrong)')
plt.tight_layout(); plt.savefig('outputs/task1_predictions.png'); plt.show()

**Task 1 observations**
* Colour (HSV) is a strong cue for flowers (yellow sunflowers/dandelions vs. white daisies vs. red/pink roses); HOG alone captures shape/texture but is less discriminative here because petals look similar in grayscale.
* Concatenating HSV + HOG usually gives the best or near-best accuracy since the two features are complementary.
* Most confusions occur between **dandelion ↔ sunflowers** (both yellow) and **roses ↔ daisy/others** when the background dominates the image.

---
# Task 2 — Camera Calibration, Distortion Correction & Perspective Transformation

**Data:** 13 checkerboard images (`left01–left14.jpg`) from the official OpenCV samples, taken with a wide-angle lens showing visible **barrel distortion**. The board has **9 × 6 internal corners**.

**Pinhole model:** a 3-D point **X** is projected to pixel **x** as  **x ∼ K [R | t] X**, where

$$K=\begin{bmatrix} f_x & s & c_x\\ 0 & f_y & c_y\\ 0&0&1\end{bmatrix},\qquad D=[k_1,k_2,p_1,p_2,k_3]$$

$k_i$ = radial distortion, $p_i$ = tangential distortion.

In [ ]:
# ---------- 2.1 Load calibration images ----------
base = 'https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/'
calib_files = []
for i in range(1, 15):
    name = f'left{i:02d}.jpg'
    try:
        calib_files.append(download(base + name, f'data/{name}'))
    except Exception:
        pass   # left10.jpg does not exist in the OpenCV repo
print(f'{len(calib_files)} calibration images:', [os.path.basename(f) for f in calib_files])

fig, axes = plt.subplots(2, 7, figsize=(18, 5))
for ax in axes.ravel(): ax.axis('off')
for ax, f in zip(axes.ravel(), calib_files):
    show(cv2.imread(f), os.path.basename(f), ax)
plt.tight_layout(); plt.show()

In [ ]:
# ---------- 2.2 Detect corners + sub-pixel refinement ----------
PATTERN = (9, 6)            # internal corners (cols, rows)
SQUARE_SIZE = 1.0           # unit size of a square (use mm for metric results)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# 3-D object points of the board: (0,0,0), (1,0,0), ..., (8,5,0)  (board plane Z = 0)
objp = np.zeros((PATTERN[0] * PATTERN[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:PATTERN[0], 0:PATTERN[1]].T.reshape(-1, 2) * SQUARE_SIZE

objpoints, imgpoints, used_files = [], [], []
vis = []
for f in calib_files:
    img = cv2.imread(f); gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    found, corners = cv2.findChessboardCorners(gray, PATTERN, None)
    if found:
        corners = corners.reshape(-1, 1, 2).astype(np.float32)
        corners_sub = cv2.cornerSubPix(gray, corners.copy(), (11, 11), (-1, -1), criteria)  # copy: refines in place
        objpoints.append(objp); imgpoints.append(corners_sub); used_files.append(f)
        shift = np.linalg.norm(corners_sub - corners, axis=2).mean()
        print(f'{os.path.basename(f)}: corners found, mean sub-pixel shift = {shift:.3f} px')
        vis.append((cv2.drawChessboardCorners(img.copy(), PATTERN, corners_sub, found), os.path.basename(f)))
    else:
        print(f'{os.path.basename(f)}: corners NOT found')

img_size = gray.shape[::-1]   # (width, height)
print(f'\nValid views: {len(objpoints)} / {len(calib_files)} | image size = {img_size}')

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (v, t) in zip(axes, vis[:4]): show(v, t + ' — detected corners', ax)
plt.tight_layout(); plt.savefig('outputs/task2_corners.png'); plt.show()

In [ ]:
# ---------- 2.3 Zoom: raw vs. sub-pixel refined corner ----------
f0 = used_files[0]; g0 = cv2.cvtColor(cv2.imread(f0), cv2.COLOR_BGR2GRAY)
_, raw = cv2.findChessboardCorners(g0, PATTERN, None)
raw = raw.reshape(-1, 1, 2)
ref = imgpoints[0]
cx, cy = ref[20, 0]; r = 12
plt.figure(figsize=(5, 5))
plt.imshow(g0, cmap='gray'); plt.xlim(cx - r, cx + r); plt.ylim(cy + r, cy - r)
plt.scatter(raw[:, 0, 0], raw[:, 0, 1], c='red', s=80, marker='x', label='findChessboardCorners')
plt.scatter(ref[:, 0, 0], ref[:, 0, 1], c='lime', s=80, marker='+', label='cornerSubPix (refined)')
plt.legend(loc='lower left'); plt.title('Sub-pixel corner refinement (zoomed)')
plt.savefig('outputs/task2_subpixel_zoom.png'); plt.show()

In [ ]:
# ---------- 2.4 Calibrate camera ----------
rms, K, D, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, img_size, None, None)

np.set_printoptions(precision=4, suppress=True)
print('Intrinsic camera matrix K:\n', K)
print(f'\nfx = {K[0,0]:.2f}, fy = {K[1,1]:.2f}, cx = {K[0,2]:.2f}, cy = {K[1,2]:.2f}')
print('\nDistortion coefficients D = [k1, k2, p1, p2, k3]:\n', D.ravel())
print(f'\nRMS re-projection error returned by calibrateCamera: {rms:.4f} px')

# Mean re-projection error (per-view, computed manually)
per_view = []
for i in range(len(objpoints)):
    proj, _ = cv2.projectPoints(objpoints[i], rvecs[i], tvecs[i], K, D)
    per_view.append(np.linalg.norm(imgpoints[i] - proj, axis=2).mean())   # mean Euclidean error (px)
mean_err = np.mean(per_view)
print(f'Mean re-projection error: {mean_err:.4f} px')

plt.figure(figsize=(10, 3.5))
plt.bar([os.path.basename(f).replace('.jpg', '') for f in used_files], per_view, color='steelblue')
plt.axhline(mean_err, color='red', ls='--', label=f'mean = {mean_err:.4f} px')
plt.ylabel('error (px)'); plt.title('Re-projection error per calibration image'); plt.legend()
plt.tight_layout(); plt.savefig('outputs/task2_reprojection_error.png'); plt.show()

In [ ]:
# ---------- 2.5 Visualise lens distortion (displacement field) ----------
w, h = img_size
gx, gy = np.meshgrid(np.linspace(0, w - 1, 20), np.linspace(0, h - 1, 15))
distorted = np.stack([gx.ravel(), gy.ravel()], 1).astype(np.float32).reshape(-1, 1, 2)
undist_pts = cv2.undistortPoints(distorted, K, D, P=K).reshape(-1, 2)
d = distorted.reshape(-1, 2)
mag = np.linalg.norm(undist_pts - d, axis=1)

plt.figure(figsize=(8, 6))
q = plt.quiver(d[:, 0], d[:, 1], undist_pts[:, 0] - d[:, 0], undist_pts[:, 1] - d[:, 1], mag,
               angles='xy', scale_units='xy', scale=1, cmap='viridis')
plt.colorbar(q, label='pixel displacement')
plt.plot(K[0, 2], K[1, 2], 'r*', ms=15, label='principal point')
plt.gca().invert_yaxis(); plt.gca().set_aspect('equal'); plt.legend()
plt.title('Lens distortion field: distorted → corrected pixel positions\n(radial, grows toward the edges = barrel distortion)')
plt.tight_layout(); plt.savefig('outputs/task2_distortion_field.png'); plt.show()

In [ ]:
# ---------- 2.6 Optimal new camera matrix + undistortion ----------
test_file = 'data/left12.jpg'
test_img = cv2.imread(test_file)

# alpha = 1 keeps all source pixels (black borders); alpha = 0 crops to valid pixels only
newK, roi = cv2.getOptimalNewCameraMatrix(K, D, img_size, alpha=1, newImgSize=img_size)
undist_full = cv2.undistort(test_img, K, D, None, newK)
x, y0, rw, rh = roi
undist_crop = undist_full[y0:y0 + rh, x:x + rw]
print('Optimised camera matrix (alpha=1):\n', newK, '\nValid ROI (x, y, w, h):', roi)

# Draw straight reference lines between outer board corners to show bending
def draw_ref_lines(img, pts):
    out = img.copy(); g = pts.reshape(PATTERN[1], PATTERN[0], 2)
    for row in [0, PATTERN[1] - 1]:
        cv2.line(out, tuple(map(int, g[row, 0])), tuple(map(int, g[row, -1])), (0, 0, 255), 2)
    for col in [0, PATTERN[0] - 1]:
        cv2.line(out, tuple(map(int, g[0, col])), tuple(map(int, g[-1, col])), (0, 0, 255), 2)
    return out

gray_t = cv2.cvtColor(test_img, cv2.COLOR_BGR2GRAY)
ok1, c1 = cv2.findChessboardCorners(gray_t, PATTERN)
ok2, c2 = cv2.findChessboardCorners(cv2.cvtColor(undist_full, cv2.COLOR_BGR2GRAY), PATTERN)
left  = draw_ref_lines(test_img, c1) if ok1 else test_img
right = draw_ref_lines(undist_full, c2) if ok2 else undist_full

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
show(left, 'Original (distorted) — straight red lines vs. curved edges', ax[0])
show(right, 'Undistorted (alpha = 1, newK)', ax[1])
show(undist_crop, 'Undistorted & cropped to valid ROI', ax[2])
plt.tight_layout(); plt.savefig('outputs/task2_undistortion.png'); plt.show()

### Perspective transformation (homography)
A homography **H** (3×3, 8 DoF) maps points of one plane to another: $\mathbf{x}' \sim H\mathbf{x}$. With four point correspondences, `cv2.getPerspectiveTransform` solves for H exactly; `cv2.warpPerspective` then resamples the image to a fronto-parallel (top-down) view.

The four source points are the **outer corners of the tilted checkerboard** in the undistorted image (detected automatically; a manual fallback is provided). The destination rectangle keeps the true 8 : 5 square ratio of the board.

In [ ]:
# ---------- 2.7 Homography: rectify the tilted board to a top-down view ----------
def order_points(pts):
    # Return points ordered TL, TR, BR, BL
    pts = np.asarray(pts, np.float32)
    s, d = pts.sum(1), np.diff(pts, axis=1).ravel()
    return np.array([pts[np.argmin(s)], pts[np.argmin(d)], pts[np.argmax(s)], pts[np.argmax(d)]], np.float32)

src_img = undist_full.copy()
ok, c = cv2.findChessboardCorners(cv2.cvtColor(src_img, cv2.COLOR_BGR2GRAY), PATTERN)
sq = 50; margin = 40                      # 50 px per square, 40 px border in the output
if ok:
    g = c.reshape(PATTERN[1], PATTERN[0], 2)
    src = order_points([g[0, 0], g[0, -1], g[-1, -1], g[-1, 0]])
    # is the 8-square (column) direction of the board horizontal or vertical in the image?
    cols_dir, rows_dir = g[0, -1] - g[0, 0], g[-1, 0] - g[0, 0]
    horizontal = abs(cols_dir[0]) > abs(rows_dir[0])
else:
    # manual fallback: four corners picked by hand on the undistorted image (TL, TR, BR, BL)
    src = order_points(np.float32([[237, 97], [415, 87], [440, 394], [210, 396]]))
    horizontal = False
nx, ny = (PATTERN[0] - 1, PATTERN[1] - 1) if horizontal else (PATTERN[1] - 1, PATTERN[0] - 1)
W, Hh = nx * sq, ny * sq                  # keeps the true square aspect ratio of the board
dst = np.float32([[margin, margin], [margin + W, margin],
                  [margin + W, margin + Hh], [margin, margin + Hh]])

H = cv2.getPerspectiveTransform(src, dst)
warped = cv2.warpPerspective(src_img, H, (W + 2 * margin, Hh + 2 * margin))
print('Source points (TL, TR, BR, BL):\n', src)
print('Destination points:\n', dst)
print('\nHomography matrix H:\n', H)

marked = src_img.copy()
cv2.polylines(marked, [src.astype(int)], True, (0, 255, 0), 2)
for i, (px, py) in enumerate(src.astype(int)):
    cv2.circle(marked, (px, py), 7, (0, 0, 255), -1)
    cv2.putText(marked, ['TL', 'TR', 'BR', 'BL'][i], (px + 8, py - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
show(marked, 'Undistorted image — selected tilted planar region', ax[0])
show(warped, 'Rectified top-down view (warpPerspective)', ax[1])
plt.tight_layout(); plt.savefig('outputs/task2_homography.png'); plt.show()

**Task 2 observations**
* The camera shows strong **barrel distortion** (negative $k_1$): straight board edges bow outward and displacement grows with distance from the principal point.
* A mean re-projection error well below **1 px** indicates an accurate calibration.
* `alpha = 1` keeps every source pixel (curved black borders appear); cropping to the ROI (or `alpha = 0`) gives a clean rectangular image at the cost of some field of view.
* After undistortion, the homography turns the obliquely viewed board into a square-celled, top-down grid — this only works correctly because lens distortion was removed first (a homography is a *linear* projective map and cannot model non-linear lens distortion).

---
# Task 3 — Video Stream Analysis: Motion Detection & Object Tracking

**Video:** OpenCV sample `vtest.avi` (fixed surveillance camera, pedestrians walking).
*(Colab runs on a remote server, so `cv2.VideoCapture(0)` cannot open your webcam there; on a local Jupyter/VS Code setup set `VIDEO_SOURCE = 0` for a live webcam.)*

**Pipeline per frame:**
1. **MOG2** background subtraction → foreground mask (255 = foreground, 127 = shadow, 0 = background)
2. **Threshold** at 200 → removes shadow pixels
3. **Morphological opening** (removes speckle noise) then **closing** (fills holes inside blobs)
4. **External contours** → filter by minimum area → bounding boxes + centroids
5. **Centroid tracker** assigns IDs and keeps the last N centroids → motion trails
6. Overlay object count + FPS → write with `cv2.VideoWriter`

In [ ]:
# ---------- 3.1 Load video & initialise MOG2 ----------
VIDEO_SOURCE = download('https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/vtest.avi', 'data/vtest.avi')

cap = cv2.VideoCapture(VIDEO_SOURCE)
assert cap.isOpened(), 'Could not open video'
fps_in = cap.get(cv2.CAP_PROP_FPS) or 25
W_v = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H_v = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
N_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Resolution: {W_v}x{H_v} | FPS: {fps_in:.1f} | Frames: {N_frames} | Duration: {N_frames / fps_in:.1f} s')
cap.release()

def make_subtractor():
    return cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=25, detectShadows=True)

In [ ]:
# ---------- 3.2 Simple centroid tracker (nearest-neighbour association) ----------
class CentroidTracker:
    def __init__(self, max_disappeared=15, max_distance=60, trail_len=30):
        self.next_id = 0
        self.objects = OrderedDict()     # id -> centroid
        self.trails = OrderedDict()      # id -> deque of recent centroids
        self.missing = OrderedDict()     # id -> frames since last seen
        self.max_disappeared, self.max_distance, self.trail_len = max_disappeared, max_distance, trail_len

    def _register(self, c):
        self.objects[self.next_id] = c
        self.trails[self.next_id] = deque([c], maxlen=self.trail_len)
        self.missing[self.next_id] = 0
        self.next_id += 1

    def _deregister(self, oid):
        for d in (self.objects, self.trails, self.missing): del d[oid]

    def update(self, centroids):
        if len(centroids) == 0:
            for oid in list(self.missing):
                self.missing[oid] += 1
                if self.missing[oid] > self.max_disappeared: self._deregister(oid)
            return self.objects
        if len(self.objects) == 0:
            for c in centroids: self._register(c)
            return self.objects

        ids = list(self.objects.keys())
        old = np.array(list(self.objects.values()), dtype=float)
        new = np.array(centroids, dtype=float)
        dist = np.linalg.norm(old[:, None, :] - new[None, :, :], axis=2)

        rows = dist.min(1).argsort(); cols = dist.argmin(1)[rows]
        used_r, used_c = set(), set()
        for r, c in zip(rows, cols):
            if r in used_r or c in used_c or dist[r, c] > self.max_distance: continue
            oid = ids[r]
            self.objects[oid] = centroids[c]; self.trails[oid].append(centroids[c]); self.missing[oid] = 0
            used_r.add(r); used_c.add(c)
        for r in set(range(len(ids))) - used_r:
            oid = ids[r]; self.missing[oid] += 1
            if self.missing[oid] > self.max_disappeared: self._deregister(oid)
        for c in set(range(len(centroids))) - used_c:
            self._register(centroids[c])
        return self.objects

def colour_for(i):
    rng = np.random.RandomState(i * 7 + 3)
    return tuple(int(v) for v in rng.randint(60, 255, 3))

In [ ]:
# ---------- 3.3 Process the video ----------
MIN_AREA = 500                                     # minimum contour area (px^2)
k_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))

cap = cv2.VideoCapture(VIDEO_SOURCE)
mog2 = make_subtractor()
tracker = CentroidTracker(max_disappeared=15, max_distance=60, trail_len=30)

raw_out = 'outputs/task3_tracking_raw.mp4'
writer = cv2.VideoWriter(raw_out, cv2.VideoWriter_fourcc(*'mp4v'), fps_in, (W_v, H_v))

SNAP_AT = 300                                      # frame used for the pipeline figure
snap, counts, fps_hist = {}, [], []
frame_idx, prev_t = 0, time.time()

while True:
    ok, frame = cap.read()
    if not ok: break
    frame_idx += 1

    # 1) foreground mask   2) remove shadows (127) -> keep only sure foreground (255)
    fg = mog2.apply(frame)
    _, fg_bin = cv2.threshold(fg, 200, 255, cv2.THRESH_BINARY)
    # 3) morphology: opening removes noise, closing fills holes; dilation merges body parts
    clean = cv2.morphologyEx(fg_bin, cv2.MORPH_OPEN, k_open, iterations=1)
    clean = cv2.morphologyEx(clean, cv2.MORPH_CLOSE, k_close, iterations=2)
    clean = cv2.dilate(clean, k_open, iterations=2)

    # 4) contours -> filter by area -> boxes + centroids
    contours, _ = cv2.findContours(clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes, cents = [], []
    for cnt in contours:
        if cv2.contourArea(cnt) < MIN_AREA: continue
        x, y_, w, h = cv2.boundingRect(cnt)
        M = cv2.moments(cnt)
        cx, cy = (int(M['m10'] / M['m00']), int(M['m01'] / M['m00'])) if M['m00'] else (x + w // 2, y_ + h // 2)
        boxes.append((x, y_, w, h)); cents.append((cx, cy))

    # 5) tracking + trails
    objects = tracker.update(cents)
    out = frame.copy()
    for (x, y_, w, h) in boxes:
        cv2.rectangle(out, (x, y_), (x + w, y_ + h), (0, 255, 0), 2)
    for oid, (cx, cy) in objects.items():
        if tracker.missing[oid] > 0: continue
        col = colour_for(oid); pts = list(tracker.trails[oid])
        for j in range(1, len(pts)):
            thick = int(1 + 3 * j / len(pts))
            cv2.line(out, pts[j - 1], pts[j], col, thick)
        cv2.circle(out, (cx, cy), 4, (0, 0, 255), -1)
        cv2.putText(out, f'ID {oid}', (cx + 6, cy - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.45, col, 2)

    # 6) telemetry overlay
    now = time.time(); fps = 1.0 / max(now - prev_t, 1e-6); prev_t = now
    fps_hist.append(fps); counts.append(len(boxes))
    fps_smooth = np.mean(fps_hist[-15:])
    cv2.rectangle(out, (0, 0), (290, 75), (0, 0, 0), -1)
    cv2.putText(out, f'Moving objects: {len(boxes)}', (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2)
    cv2.putText(out, f'FPS: {fps_smooth:.1f}', (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 0), 2)
    cv2.putText(out, f'Frame: {frame_idx}/{N_frames}', (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    writer.write(out)

    if frame_idx == SNAP_AT:
        snap = dict(frame=frame.copy(), fg=fg.copy(), fg_bin=fg_bin.copy(), clean=clean.copy(), out=out.copy())

cap.release(); writer.release()
print(f'Processed {frame_idx} frames | mean processing FPS = {np.mean(fps_hist[5:]):.1f}')
print(f'Average moving objects per frame = {np.mean(counts):.2f} | max = {max(counts)}')
print(f'Total unique track IDs assigned = {tracker.next_id}')
print('Saved:', raw_out)

In [ ]:
# ---------- 3.4 Pipeline stages for one frame ----------
fig, ax = plt.subplots(2, 3, figsize=(18, 9))
show(snap['frame'], f'1. Input frame #{SNAP_AT}', ax[0, 0])
show(snap['fg'], '2. MOG2 mask (white = fg, gray = shadow)', ax[0, 1])
show(snap['fg_bin'], '3. Thresholded (shadows removed)', ax[0, 2])
show(snap['clean'], '4. After opening + closing + dilation', ax[1, 0])
show(snap['out'], '5. Boxes, centroids, IDs, trails, telemetry', ax[1, 1])
ax[1, 2].plot(counts, lw=1); ax[1, 2].set_title('Moving-object count per frame')
ax[1, 2].set_xlabel('frame'); ax[1, 2].set_ylabel('count'); ax[1, 2].grid(alpha=.3)
plt.tight_layout(); plt.savefig('outputs/task3_pipeline.png'); plt.show()

In [ ]:
# ---------- 3.5 Several annotated output frames ----------
cap = cv2.VideoCapture(raw_out)
picks = [80, 200, 350, 500, 650, N_frames - 5]
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, p in zip(axes.ravel(), picks):
    cap.set(cv2.CAP_PROP_POS_FRAMES, p); ok, fr = cap.read()
    if ok: show(fr, f'frame {p}', ax)
cap.release()
plt.tight_layout(); plt.savefig('outputs/task3_frames.png'); plt.show()

In [ ]:
# ---------- 3.6 Convert to H.264 and play inline ----------
final_out = 'outputs/task3_tracking_output.mp4'
!ffmpeg -y -loglevel error -i {raw_out} -vcodec libx264 -preset veryfast -crf 26 -pix_fmt yuv420p {final_out}
print('Output video:', final_out, f'({os.path.getsize(final_out) / 1e6:.1f} MB)')

data = base64.b64encode(open(final_out, 'rb').read()).decode()
display(HTML(f'<video width="720" controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))

**Task 3 observations**
* MOG2 models every pixel as a mixture of Gaussians and adapts over time, so it handles gradual lighting change; the first frames are noisy while the model learns the background.
* Thresholding at 200 drops MOG2's shadow label (127), preventing shadows from enlarging or merging boxes.
* Opening removes isolated noise pixels; closing joins fragmented body parts into one blob. The `MIN_AREA` filter removes remaining small blobs.
* The nearest-neighbour centroid tracker keeps consistent IDs while people are separated; IDs can switch when people overlap or walk in groups (occlusion), a known limitation of centroid-only tracking (Kalman/SORT/DeepSORT address this).

In [ ]:
# ---------- Download all outputs (figures + video) ----------
!cd outputs && zip -q -r ../lab08_outputs.zip . && cd ..
print(sorted(os.listdir('outputs')))
try:
    from google.colab import files
    files.download('lab08_outputs.zip')
except Exception:
    print('Not running in Colab — outputs are in ./outputs and lab08_outputs.zip')

---
## Conclusion
* **Task 1:** Built a complete classical image-classification pipeline (resize → stratified split → HSV/HOG features → SVM/Random Forest) and evaluated it with accuracy, a classification report and confusion matrices.
* **Task 2:** Calibrated a camera from 13 checkerboard views (sub-pixel corners), obtained **K**, **D** and a sub-pixel mean re-projection error, visualised and removed barrel distortion, and rectified a tilted plane with a homography.
* **Task 3:** Implemented a real-time motion analytics pipeline (MOG2 + shadow removal + morphology + contours + centroid tracking) with trails, object count and FPS overlay, exported as an MP4 video.